# 悲观错误剪枝（Pessimistic Error Pruning, PEP）

## 讲解 + 可执行计算实例

本 Notebook 面向想真正理解 C4.5 悲观剪枝的读者。结构是：**先讲清概念，再用 Python 实例验证**。

### 目录
1. PEP 要解决什么问题
2. 统计学视角：为什么观察到的错误率是一个随机变量？
3. 两个必须区分的量：观察错误率与悲观错误率
4. 悲观错误率怎么算：二项分布置信上界
5. confidence 参数的含义与方向
6. 剪枝决策：保留子树还是剪成叶节点
7. 完整剪枝示例
8. confidence 敏感性分析
9. 优点、局限与小结
10. 附注：关于 RapidMiner 等软件中的参数配置

In [ ]:
import numpy as np
from scipy.stats import beta, norm
import pandas as pd
import matplotlib.pyplot as plt

# --- 中文字体设置 ---
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

---
## 1. PEP 要解决什么问题

决策树在训练集上生长时，某个叶节点可能只覆盖很少的样本。这一次观察到的错误率，无论具体是多少，都只是**有限样本上的一次观测结果**。

如果直接拿这一次观察到的错误率去做剪枝判断，决策就会被这次观测的偶然性左右——可能该剪的没剪，也可能不该剪的剪了。

PEP 的应对方式是：**不把这一次观察到的错误率直接当真，而是对它做保守修正，估出一个偏高的“上界”，用这个悲观估计去比较剪枝前后的代价。**

---
## 2. 统计学视角：为什么观察到的错误率是一个随机变量？

在训练集上，某个叶节点的**观察错误率** $\hat{p} = E/N$ 是一个**随机变量**：它取决于这一次抽样恰好抽到了哪些样本。换一批训练数据，数出来的 $E$ 和 $N$ 就会变，$\hat{p}$ 也随之变化。

根据统计学原理：
- 给定真实错误概率 $p$ 时，错误数 $E$ 服从二项分布 $\text{Binomial}(N, p)$。
- 因此，观察错误率 $\hat{p}$ 的方差近似为 $\frac{p(1-p)}{N}$。

**这个方差公式极其重要**：它告诉了我们，随着样本量 $N$ 的减小，方差会急剧变大。也就是说，小样本下“观察到的错误率”波动极大，单次观测值偏离真实水平的概率极高。

这正是 PEP 悲观修正的根源：**既然单次观察的随机误差极大（方差大），那为了保险，我干脆把你往最坏的方向（上界）去估。**

In [ ]:
# 演示：假设真实错误率 p 为 0.2，观察错误率的方差随 N 的变化
p = 0.2
Ns = [10, 30, 100, 500]
variances = [(p * (1 - p)) / N for N in Ns]

plt.figure(figsize=(8, 4))
plt.plot(Ns, variances, marker='o', color='purple', linestyle='-')
for n, v in zip(Ns, variances):
    plt.text(n, v + 0.001, f"{v:.4f}", ha='center', va='bottom')
plt.title('观察错误率 (E/N) 的方差随样本量 N 的变化')
plt.xlabel('样本量 N')
plt.ylabel('方差 V(p_hat) = p(1-p)/N')
plt.grid(alpha=0.3)
plt.show()

**观察**：当 $N=10$ 时，观察错误率的方差高达 0.016，意味着单次观测结果的波动极大；而当 $N=500$ 时，方差趋近于 0。这种统计学特征，是理解后续所有推导的基石。

---
## 3. 两个必须区分的量：观察错误率与悲观错误率

对每个叶节点，记样本数为 $N$，错误数为 $E$。

**观察错误率** $\hat{p} = E/N$：是一个**随机变量**。它是训练集上这一次直接数出来的结果，只是单次观测。

**悲观错误率**：记作 $U$，是未来真实错误率 $p$ 的单侧置信上界。**为了确保它是上界（即 $U \ge \hat{p}$）**，我们取的是 $1 - \text{confidence}$ 的分位数，满足：

$$P(p \le U \mid E, N) = 1 - \text{confidence}$$

其中 confidence 是人为设定的参数，C4.5 经典实现通常取 **0.25**（意味着我们要 75% 的把握覆盖真实值）。$U$ 永远不小于观察错误率，它是刻意往坏里估的结果。

In [ ]:
def pessimistic_error_rate(E, N, confidence=0.25):
    """二项分布精确置信上界。"""
    if N == 0:
        return 0.0
    # 注意：我们取 1 - confidence 的分位数，以确保它是上界
    return beta.ppf(1 - confidence, E + 1, N - E + 1)

# 控制变量：观察错误率固定
test_cases = [
    (2, 10, '观察错误率=0.2'), 
    (20, 100, '观察错误率=0.2'), 
    (0, 10, '观察错误率=0.0'),
    (0, 100, '观察错误率=0.0')
]

rows = []
for E, N, label in test_cases:
    rows.append({
        '组别': label,
        'E (错误数)': E, 
        'N (样本数)': N,
        '观察错误率 E/N': E/N,
        '悲观错误率 U': pessimistic_error_rate(E, N, 0.25)
    })
    
df = pd.DataFrame(rows)

# 画图直观对比（将整体高度从 4 调高到 5）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 统一配色：亮灰色 和 亮蓝色
color_obs = 'silver'
color_pep = 'dodgerblue'
width = 0.35

# 左图：观察错误率=0.2
subset1 = df[df['组别'] == '观察错误率=0.2']
x1 = np.arange(len(subset1))
rects1_obs = axes[0].bar(x1 - width/2, subset1['观察错误率 E/N'], width, label='观察错误率', color=color_obs)
rects1_pep = axes[0].bar(x1 + width/2, subset1['悲观错误率 U'], width, label='悲观错误率 (U)', color=color_pep)

# 添加数值标签
for rect in rects1_obs:
    height = rect.get_height()
    axes[0].annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, color='dimgray')
for rect in rects1_pep:
    height = rect.get_height()
    axes[0].annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, color='dodgerblue')

axes[0].set_xticks(x1)
axes[0].set_xticklabels([f"N={n}" for n in subset1['N (样本数)']])
axes[0].set_title('观察错误率均为 0.2 时的修正差异')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
# 动态设置左图 Y 轴上限，留出 15% 的顶部空间
axes[0].set_ylim(0, max(subset1['悲观错误率 U']) * 1.15)


# 右图：观察错误率=0
subset2 = df[df['组别'] == '观察错误率=0.0']
x2 = np.arange(len(subset2))
rects2_obs = axes[1].bar(x2 - width/2, subset2['观察错误率 E/N'], width, label='观察错误率', color=color_obs)
rects2_pep = axes[1].bar(x2 + width/2, subset2['悲观错误率 U'], width, label='悲观错误率 (U)', color=color_pep)

# 添加数值标签
for rect in rects2_obs:
    height = rect.get_height()
    axes[1].annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, color='dimgray')
for rect in rects2_pep:
    height = rect.get_height()
    axes[1].annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, color='dodgerblue')

axes[1].set_xticks(x2)
axes[1].set_xticklabels([f"N={n}" for n in subset2['N (样本数)']])
axes[1].set_title('观察错误率均为 0.0 时的修正差异')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
# 动态设置右图 Y 轴上限，留出 15% 的顶部空间
axes[1].set_ylim(0, max(subset2['悲观错误率 U']) * 1.15)

plt.tight_layout()
plt.show()

df.round(4)

**观察**：
- **样本量越小，修正越狠**：在左图中，样本量为 10 时，悲观错误率被往上抬到了约 0.33；当样本量达到 100 时，悲观错误率只被微调到约 0.23。这说明样本量越小，算法越不信任这一次观测结果，因此施加的悲观惩罚越重。
- **极端情况下的欺骗性**：对比左右两图可以发现，当真实情况完全正确（观察错误率为 0/10）时，算法不仅没有完全信任，反而直接将其拔高到了 0.12，展示了悲观剪枝对抗“小样本完美假象”的能力。
- **大样本下的信任**：值得注意的是，当 (N=100) 时，对于 0/100 这种极端完美的数据，算法的修正幅度急剧缩小（仅修正到 0.01），甚至在绝对修正量上比 20/100（修正到 0.23）还要小。这说明**当样本量足够大时，算法会对极端（零差错）的数据产生真正的信任。**

---
## 4. 悲观错误率怎么算：二项分布置信上界

把 $N$ 个样本看作 $N$ 次独立试验，每次以未知概率 $p$ 出错。在贝叶斯视角下，观察到 $E$ 个错误后，$p$ 的后验分布（假设均匀先验）为 $\text{Beta}(E+1, N-E+1)$。

为了得到一个“悲观上界”，置信上界 $U$ 就是该 Beta 分布的 **$1 - \text{confidence}$ 分位数**：

$$U = \text{Beta}_{\text{ppf}}(1 - \text{confidence};\ E+1,\ N-E+1)$$

### 实例：直观展示置信上界怎么切出来的
我们直接画出真实错误率 $p$ 的后验分布曲线，并标出“置信上界”是如何通过面积（概率）切分出来的。

In [ ]:
# --- 设定参数 ---
E = 0       # 观察到的错误数
N = 10      # 样本数
conf = 0.25 # 置信水平

# --- 计算上界 U ---
U = beta.ppf(1 - conf, E + 1, N - E + 1)

# --- 绘图 ---
# 使用更密集的点，让曲线和阴影分界线更平滑
p_vals = np.linspace(0, 0.8, 1000)
pdf_vals = beta.pdf(p_vals, E + 1, N - E + 1)

fig, ax = plt.subplots(figsize=(10, 6))

# 1. 画真实错误率 p 的后验分布曲线
ax.plot(p_vals, pdf_vals, 'b-', label='真实错误率 p 的分布')

# 2. 阴影区域：p <= U 的部分（覆盖 1 - confidence 面积）
ax.fill_between(p_vals, pdf_vals, where=(p_vals <= U), 
                color='skyblue', alpha=0.6, 
                label=f'面积 = {(1-conf)*100}% (置信水平)')

# 3. 阴影区域：p > U 的部分（覆盖 confidence 面积）
ax.fill_between(p_vals, pdf_vals, where=(p_vals > U), 
                color='salmon', alpha=0.3, 
                label=f'有 {conf*100}% 概率真实值比 U 更大')

# 4. 标注观察到的错误率 E/N = 0
ax.axvline(E/N, color='gray', linestyle='--', label=f'观察错误率 (E/N = {E/N})')

# 5. 标注置信上界 U
ax.axvline(U, color='red', linestyle='-', linewidth=2, label=f'悲观错误率上界 (U ≈ {U:.3f})')

# 设置坐标轴
ax.set_xlim(0, 0.8)
ax.set_ylim(0, max(pdf_vals) * 1.2)
ax.set_title(f'悲观错误率的计算过程: E={E}, N={N}, confidence={conf}', fontsize=14)
ax.set_xlabel('真实错误率 p', fontsize=12)
ax.set_ylabel('概率密度', fontsize=12)

# 6. 添加注释
ax.annotate(f'因为真实错误率 p ≤ {U:.3f} 的把握\n有 {(1-conf)*100}%，\n所以 {U:.3f} 是悲观上界', 
            xy=(U, beta.pdf(U, E+1, N-E+1)/2), 
            xytext=(0.35, max(pdf_vals)*0.6),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
            fontsize=12, 
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))

ax.legend(loc='upper right', fontsize=11)
ax.grid(alpha=0.3)
plt.show()

**观察**：
- 灰色虚线处 $E/N = 0$，是“观察到的完美情况”。
- 红色实线处 $U \approx 0.206$ 是算出来的悲观上界（完美契合了经典教材中的数值）。
- 浅蓝色面积占了 75%，代表“真实错误率小于等于 $U$”的把握有 75%；红色面积占 25%。
- 所谓“悲观”，就是我们强行把错误率估值从 0 拉到了这个足以覆盖 75% 可能性的上界 $U$。

### 实例：精确二项分布 vs 正态近似
实际实现中，也常用**正态近似**来简化计算，公式为：
$$U \approx \frac{E}{N} + Z_{1-\text{confidence}} \cdot \sqrt{\frac{(E/N)(1-E/N)}{N}}$$
下面比较两者的差异。

In [ ]:
def pessimistic_error_rate_normal(E, N, confidence=0.25):
    if N == 0:
        return 0.0
    p_hat = E / N
    # 同样取 1 - confidence 对应的 Z 值
    z = norm.ppf(1 - confidence)
    return p_hat + z * np.sqrt(p_hat * (1 - p_hat) / N)

rows = []
for E, N in [(0, 10), (1, 10), (2, 10), (0, 20), (5, 20), (10, 100)]:
    for c in [0.25, 0.95]:
        rows.append({
            'E': E, 'N': N, 'confidence': c,
            '精确二项上界': pessimistic_error_rate(E, N, c),
            '正态近似上界': pessimistic_error_rate_normal(E, N, c)
        })
pd.DataFrame(rows).round(4)

**观察**：
- $E=0, N=10$，confidence=0.25 时，精确二项上界约 **0.206**，正态近似算出来也是 **0.206**。
- confidence 从 0.25 升到 0.95 时，修正幅度变小，上界显著下降。

---
## 5. confidence 参数的含义与方向

confidence 控制的是“上界 $U$ 覆盖真实错误率的把握”，**不是**“观察错误率为真的概率”。

- confidence 低（如 0.25）：对“上界覆盖真实值”的把握要求低（只要 75% 覆盖即可），上界可以取得很高，估计很悲观。
- confidence 高（如 0.95）：要求高（要 95% 覆盖），上界被压低，估计接近观察值。

方向是：**confidence 低 → 不信任这一次观察 → 上界高 → 悲观 → 剪枝激进。**

### 实例：固定 $E=0, N=10$，扫描 confidence

In [ ]:
E, N = 0, 10
confs = np.linspace(0.01, 0.99, 99)
U_vals = [pessimistic_error_rate(E, N, c) for c in confs]

plt.figure(figsize=(9, 5))
plt.plot(confs, U_vals)
plt.axhline(E/N, color='gray', linestyle='--', label='observed error rate = 0')
plt.xlabel('confidence')
plt.ylabel('pessimistic error rate U')
plt.title(f'U vs confidence for E={E}, N={N}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for c in [0.1, 0.25, 0.5, 0.95]:
    print(f'confidence={c}: U={pessimistic_error_rate(E, N, c):.4f}')

**观察**：confidence 越低，$U$ 越高，估计越悲观；$U$ 始终不小于观察错误率 0。这验证了“低 confidence → 上界高 → 悲观”。

---
## 6. 剪枝决策：保留子树还是剪成叶节点

PEP 对每个内部节点比较两个方案。

**方案 A：保留当前子树**

对子树下每一个叶节点 $i$，用其 $(E_i, N_i)$ 和 confidence 算出悲观错误率 $U_i$，换算为悲观错误数 $N_i \times U_i$。子树总代价：

$$\text{代价}_A = \sum_i N_i U_i$$

**方案 B：将子树替换为一个叶节点**

合并后 $N_{\text{total}} = \sum N_i$，$E_{\text{total}} = \sum E_i$，用同一 confidence 算出 $U_{\text{total}}$，代价：

$$\text{代价}_B = N_{\text{total}} \times U_{\text{total}}$$

**决策规则**：若 $\text{代价}_B \le \text{代价}_A$，剪掉子树，用叶节点替代；否则保留。

### 实例：实现剪枝决策函数

In [ ]:
def pep_prune_decision(leaves, confidence=0.25, use_normal=False):
    """
    leaves: list of (E, N)
    返回方案 A/B 代价与是否剪枝。
    """
    f = pessimistic_error_rate_normal if use_normal else pessimistic_error_rate
    cost_A = sum(N * f(E, N, confidence) for E, N in leaves)
    E_total = sum(E for E, N in leaves)
    N_total = sum(N for E, N in leaves)
    cost_B = N_total * f(E_total, N_total, confidence)
    return {
        '代价A_保留子树': cost_A,
        '代价B_剪成叶节点': cost_B,
        '剪枝': cost_B <= cost_A
    }

---
## 7. 完整剪枝示例

为了让你更直观地看到剪枝效果，我们模拟三个经典场景。
在每个场景中，我们比较将一棵子树**保留**（分别看各个叶节点）与**剪枝**（合并成一个单一叶节点）的代价。

In [ ]:
def print_prune_result(name, leaves, confidence=0.25):
    res = pep_prune_decision(leaves, confidence=confidence)
    E_total = sum(E for E, N in leaves)
    N_total = sum(N for E, N in leaves)
    
    print(f"=== {name} ===")
    print(f"子树包含 {len(leaves)} 个叶节点:")
    for i, (E, N) in enumerate(leaves):
        print(f"  叶节点 {i+1}: 观察错误 {E}/{N} ({E/N:.2f}) -> 悲观错误率 {pessimistic_error_rate(E, N, confidence):.3f}")
    print(f"合并为单个叶节点: 观察错误 {E_total}/{N_total} ({E_total/N_total:.2f})")
    print(f"--> 方案 A (保留) 总悲观代价: {res['代价A_保留子树']:.3f}")
    print(f"--> 方案 B (剪枝) 总悲观代价: {res['代价B_剪成叶节点']:.3f}")
    print(f"决策结果: {'【剪枝】砍掉子树' if res['剪枝'] else '【保留】不剪枝'}\n")

# 场景 1：典型的过度拟合（叶子多，样本少且都是 0 错误）
leaves_1 = [(0, 6), (0, 9), (0, 1)]
print_prune_result("场景 1：过度拟合的子树", leaves_1)

# 场景 2：错误率偏高的子树（叶子多，且各叶节点含有错误）
leaves_2 = [(1, 5), (2, 5), (1, 5)]
print_prune_result("场景 2：错误率偏高的子树", leaves_2)

# 场景 3：大样本下的可信子树（子树大且样本极大，悲观修正少）
leaves_3 = [(10, 100), (15, 100)]
print_prune_result("场景 3：大样本可信子树", leaves_3)

**场景解读**：
- **场景 1**：三个叶节点样本极少（6, 9, 1）且全对。PEP 认为这不可信，把各自的错误率大幅抬高（比如 1 个样本被抬高到 0.75），导致保留子树的代价（3.25）远高于剪枝合并后的代价（2.51），从而**果断剪枝**。
- **场景 2**：每个叶节点都在犯错误，保留它们不仅没有降低错误，反而因为叶子多增加了累加惩罚，因此也会被**果断剪枝**。
- **场景 3**：样本量极大（各 100 个），算法对这两个叶节点非常信任，悲观修正幅度很小。此时“保留子树”的代价（21.6）小于“合并剪枝”的代价（26.8），因此系统判断**不应剪枝**，保留这棵大子树。

---
## 8. confidence 敏感性分析

固定上述子树，扫描 confidence 从 0.01 到 0.99，观察 $A$、$B$ 与决策变化。

In [ ]:
confs = np.linspace(0.01, 0.99, 99)
A_vals, B_vals, decisions = [], [], []

for c in confs:
    res = pep_prune_decision(leaves_1, confidence=c)
    A_vals.append(res['代价A_保留子树'])
    B_vals.append(res['代价B_剪成叶节点'])
    decisions.append(res['剪枝'])

plt.figure(figsize=(9, 5))
plt.plot(confs, A_vals, label='Cost A: keep subtree')
plt.plot(confs, B_vals, label='Cost B: prune to leaf')
plt.xlabel('confidence')
plt.ylabel('pessimistic error cost')
plt.title('PEP cost vs confidence')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

flip = None
for i in range(1, len(confs)):
    if decisions[i] != decisions[i-1]:
        flip = confs[i]
        break
print('决策翻转点 confidence ≈', flip)
print('低 confidence 时是否剪枝:', decisions[0])
print('高 confidence 时是否剪枝:', decisions[-1])

**观察**：confidence 较低时，方案 A 因多个叶节点累加悲观修正而膨胀，容易 $B \le A$，剪枝；confidence 升高，修正变弱，A 下降，可能不再剪枝。这直观展示了“confidence 越低，剪枝越激进”。

---
## 9. 优点、局限与小结

### 优点
- 不需要独立验证集，数据有限时可用。
- 计算高效，剪枝较快。

### 局限
- 把分类错误数套用二项分布置信区间是启发式近似，统计基础受质疑。
- confidence 与剪枝强度的反向关系反直觉，容易误用。
- 有观点认为这类方法可能偏向剪枝不足。

### 一句话总结

观察到的错误率是一个**随机变量**，方差随样本减小而剧增。PEP 为了抵抗这种不确定性，用给定 confidence 和 $(E,N)$ 按二项分布求真实错误率的保守上界 $U$，以“保留子树时各叶节点悲观错误数之和”与“合并成一个叶节点后的悲观错误数”比较决定剪枝。confidence 越低，上界越高，估得越悲观，剪枝越激进。

---
## 10. 附注：关于 RapidMiner 等软件中的参数配置


如果在实际建模中使用了 RapidMiner 或其他工业级软件，你可能会发现前文的结论与软件中的表现**完全相反**。这是因为软件底层的参数命名逻辑与前文推导时采用的视角不同。

### 1. 软件里的 confidence 和前文公式里的 confidence 是什么关系？
前文推导 PEP 时，公式写为：
$$U = \text{Beta}_{\text{ppf}}(1 - \text{confidence};\ E+1,\ N-E+1)$$

前文公式里的 $\text{confidence}$ 代表的是“**置信水平**”（即要求上界覆盖真实值的把握）。

而在 RapidMiner 等软件中，参数 `confidence` 实际上代表的是“**容忍出错的风险率**”（类似于统计假设检验中的显著性水平 $\alpha$）。两者恰好是互补关系：
$$\text{软件中的 confidence} = 1 - \text{前文公式中的 confidence}$$

举例来说：
- 当软件设 `confidence` 为 **0.1** 时，相当于前文公式里的 $1 - 0.1 = 0.9$（要求 90% 的把握）。此时算出的悲观惩罚较轻，保留了较多枝叶。
- 当软件设 `confidence` 为 **0.5** 时，相当于前文公式里的 $1 - 0.5 = 0.5$（只有 50% 的把握）。这种极度的不信任会导致小样本节点的惩罚项被剧烈放大，算法认为保留复杂子树毫无意义，从而大幅剪枝。

所以，**软件中的 `confidence` 越大，等价于前文公式中的置信水平越低，剪枝反而越激进**。

### 2. 为什么其取值范围被限制在 $1.0\text{E-}7$ 到 $0.5$ 之间？
- **上限 0.5**：PEP 要求算出的值必须是真实错误率的“上界”。若软件中的 `confidence` 大于 0.5，则前文公式中的 $(1 - \text{confidence})$ 会小于 0.5，此时算出的区间甚至会低于观察错误率，违背了悲观原则。0.5 是数学上最极端的剪枝阈值。
- **下限 $1.0\text{E-}7$**：趋近于 0 意味着要求近乎 100% 的置信度（即前文公式里的 $(1 - \text{confidence})$ 趋近于 1）。为防止工程实现中出现除以 0 或数值溢出，通常设定一个极小值，它在效果上等同于“几乎不剪枝”。

### 3. 预剪枝与后剪枝协同
RapidMiner 同时提供预剪枝与后剪枝选项，它们并不冲突：前者在树生长时利用增益等条件提前阻止分裂；后者在树长成后进行全局回看剪枝。两者结合能有效弥补预剪枝的缺陷。